# Projeto — Análise Exploratória de Dados de Vendas

**Período analisado:** 1º de janeiro a 31 de dezembro de 2023  
**Fonte:** `vendas_supermercado_com_cliente.csv`  
Os gráficos foram construídos com **Seaborn** e **Matplotlib**; as conclusões calculadas aparecem junto às visualizações.

Este notebook responde às dez perguntas do projeto. Cada seção apresenta o critério adotado e uma tabela ou gráfico produzido a partir dos dados. Valores monetários são apresentados na moeda/unidade registrada no arquivo, que não especifica moeda.

> **Limitação da base:** cada linha representa um produto em uma data, sem código de transação, cliente ou loja. Assim, “ticket médio” é tratado como valor médio de venda por registro (linha), não como valor médio de uma cesta. As associações observadas não demonstram causalidade.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
from IPython.display import display

ARQUIVO = Path("vendas_supermercado_com_cliente.csv")
df = pd.read_csv(ARQUIVO, encoding="utf-8-sig")
df["Data"] = pd.to_datetime(df["Data"], errors="coerce")
df = df.sort_values("Data").reset_index(drop=True)
df["Mes"] = df["Data"].dt.month
df["Mes_nome"] = df["Data"].dt.month_name(locale="pt_BR.utf8") if False else df["Data"].dt.month.map({1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"})
df["Dia_semana"] = df["Data"].dt.dayofweek
nomes_dias = {0:"Segunda",1:"Terça",2:"Quarta",3:"Quinta",4:"Sexta",5:"Sábado",6:"Domingo"}
df["Dia_semana_nome"] = df["Dia_semana"].map(nomes_dias)
df["Promocao"] = df["Desconto"].gt(0)
print(f"Registros: {len(df):,} | Produtos: {df['Produto'].nunique()} | Categorias: {df['Categoria'].nunique()}")
print(f"Período: {df['Data'].min():%d/%m/%Y} a {df['Data'].max():%d/%m/%Y} | Valores ausentes: {int(df.isna().sum().sum())}")
display(df.head())

## 1. Categorias mais vendidas e meses de pico

“Mais vendidas” é medido pela quantidade total de unidades. O pico mensal de cada categoria também usa unidades.

In [ ]:
cat_total = df.groupby("Categoria")["Quantidade"].sum().sort_values(ascending=False)
cat_mes = df.groupby(["Categoria", "Mes_nome"], as_index=False)["Quantidade"].sum()
mes_ordem = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
cat_mes["Mes_nome"] = pd.Categorical(cat_mes["Mes_nome"], categories=mes_ordem, ordered=True)
picos = cat_mes.loc[cat_mes.groupby("Categoria")["Quantidade"].idxmax()].sort_values("Quantidade", ascending=False)
res = picos.merge(cat_total.rename("Unidades_ano"), on="Categoria").sort_values("Unidades_ano", ascending=False)
display(res.rename(columns={"Quantidade":"Unidades_no_pico","Mes_nome":"Mes_pico"}).reset_index(drop=True))
print("Produtos com maior quantidade anual:")
top_prod = df.groupby("Produto")["Quantidade"].sum().sort_values(ascending=False).head(10)
display(top_prod.rename("Unidades").to_frame())
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=cat_total.head(10).values,y=cat_total.head(10).index,ax=ax,color="#3B82A0")
ax.set(title="Categorias líderes em unidades vendidas",xlabel="Unidades no ano",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {cat_total.index[0]} liderou com {cat_total.iloc[0]:,} unidades; o mês de pico varia por categoria conforme a tabela acima.")

## 2. Média de vendas diárias por categoria

Somam-se os valores de venda por categoria em cada data e calcula-se a média diária ao longo do período.

In [ ]:
por_dia_cat = df.groupby(["Data", "Categoria"])["Total_Venda"].sum().unstack(fill_value=0)
media_diaria_cat = por_dia_cat.mean().sort_values(ascending=False)
display(media_diaria_cat.rename("Venda_media_diaria").to_frame())
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=media_diaria_cat.values,y=media_diaria_cat.index,ax=ax,color="#4E79A7")
ax.set(title="Venda média diária por categoria",xlabel="Valor médio por dia",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {media_diaria_cat.idxmax()} teve a maior venda média diária ({media_diaria_cat.max():,.2f}).")


## 3. Sazonalidade dos produtos

A sazonalidade é estimada pelo coeficiente de variação (CV = desvio-padrão/média) das quantidades mensais de cada produto. CV alto indica maior variação relativa entre meses; com apenas um ano, não comprova um padrão sazonal recorrente.

In [ ]:
prod_mes = df.groupby(["Produto", "Mes"])["Quantidade"].sum().unstack()
cv = prod_mes.std(axis=1, ddof=0) / prod_mes.mean(axis=1).replace(0, np.nan)
sazonal = pd.DataFrame({"CV_mensal": cv, "Menor_mes": prod_mes.idxmin(axis=1), "Maior_mes": prod_mes.idxmax(axis=1)})
sazonal["Menor_mes"] = sazonal["Menor_mes"].map(dict(enumerate(mes_ordem, start=1)) if False else {i+1:n for i,n in enumerate(mes_ordem)})
sazonal["Maior_mes"] = pd.DataFrame({"Mes": prod_mes.idxmax(axis=1)}).Mes.map({i+1:n for i,n in enumerate(mes_ordem)})
display(sazonal.sort_values("CV_mensal", ascending=False).head(15))
top_cv=sazonal.sort_values("CV_mensal",ascending=False).head(10).sort_values("CV_mensal")
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=top_cv.CV_mensal.values,y=top_cv.index,ax=ax,color="#F28E2B")
ax.set(title="Produtos com maior variação relativa mensal",xlabel="Coeficiente de variação mensal",ylabel=""); plt.tight_layout(); plt.show()
print(f"Conclusão: {sazonal.CV_mensal.idxmax()} teve o maior CV mensal ({sazonal.CV_mensal.max():.1%}). Com apenas um ano, isso mede variação dentro do ano, não recorrência sazonal.")

## 4. Faixas etárias com maior contribuição às vendas

Faixas conforme o enunciado: jovem (18–35), adulto (36–59) e idoso (60 anos ou mais). A faixa idoso inclui 60 anos para não deixar essa idade sem classificação.

In [ ]:
df["Faixa_etaria"] = pd.cut(df["Idade"], bins=[17,35,59,np.inf], labels=["Jovem (18–35)","Adulto (36–59)","Idoso (60+)"])
faixa_vendas = df.groupby("Faixa_etaria", observed=False).agg(Venda_total=("Total_Venda","sum"), Participacao=("Total_Venda",lambda x: x.sum()/df["Total_Venda"].sum()), Registros=("Total_Venda","size"), Idade_media=("Idade","mean"))
display(faixa_vendas)
fig,ax=plt.subplots(figsize=(7,4)); sns.barplot(x=faixa_vendas.index.astype(str),y=faixa_vendas.Venda_total.values,ax=ax)
ax.set(title="Vendas totais por faixa etária",ylabel="Valor de venda",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
lider_faixa=faixa_vendas.Venda_total.idxmax(); print(f"Conclusão: {lider_faixa} contribuiu mais para a receita ({faixa_vendas.loc[lider_faixa,'Participacao']:.1%} do total).")

## 5. Relação entre idade e categoria mais vendida

Para cada faixa, a categoria líder é definida pela soma de unidades; a tabela também mostra sua participação nas unidades da faixa.

In [ ]:
faixa_cat = df.groupby(["Faixa_etaria","Categoria"], observed=False).agg(Unidades=("Quantidade","sum"), Venda=("Total_Venda","sum")).reset_index()
faixa_cat["Participacao_unidades_faixa"] = faixa_cat["Unidades"] / faixa_cat.groupby("Faixa_etaria", observed=False)["Unidades"].transform("sum")
lideres = faixa_cat.loc[faixa_cat.groupby("Faixa_etaria", observed=False)["Unidades"].idxmax()].sort_values("Faixa_etaria")
display(lideres)
mat=faixa_cat.pivot(index="Categoria",columns="Faixa_etaria",values="Unidades").fillna(0)
fig,ax=plt.subplots(figsize=(11,5)); long=mat.reset_index().melt(id_vars="Categoria",var_name="Faixa_etaria",value_name="Unidades")
sns.barplot(data=long,x="Categoria",y="Unidades",hue="Faixa_etaria",ax=ax)
ax.set(title="Unidades por categoria e faixa etária",ylabel="Unidades",xlabel=""); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()
print("Conclusão — categoria líder por faixa: " + "; ".join(f"{r.Faixa_etaria}: {r.Categoria}" for r in lideres.itertuples()))

## 6. Distribuição das vendas por dia da semana

Compara-se a média de venda diária total por dia da semana, evitando que meses com mais ocorrências de um dia distorçam o total.

In [ ]:
dia = df.groupby(["Data","Dia_semana_nome"])["Total_Venda"].sum().reset_index()
ordem_dias = ["Segunda","Terça","Quarta","Quinta","Sexta","Sábado","Domingo"]
dist_dia = dia.groupby("Dia_semana_nome")["Total_Venda"].agg(Media_diaria="mean", Total="sum", Dias="count").reindex(ordem_dias)
display(dist_dia)
fig,ax=plt.subplots(figsize=(9,4)); sns.barplot(x=dist_dia.index,y=dist_dia.Media_diaria.values,ax=ax,color="#76B7B2")
ax.set(title="Venda média diária por dia da semana",ylabel="Valor médio por dia",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
mais_forte=dist_dia.Media_diaria.idxmax(); mais_fraco=dist_dia.Media_diaria.idxmin()
print(f"Conclusão: {mais_forte} teve a maior média diária ({dist_dia.loc[mais_forte,'Media_diaria']:,.2f}) e {mais_fraco} a menor ({dist_dia.loc[mais_fraco,'Media_diaria']:,.2f}).")


## 7. Vendas mensais antes e durante a Black Friday

Como a Black Friday de 2023 ocorreu em 24 de novembro, “durante” é representado por novembro completo; janeiro a outubro representam o período anterior. A média diária ajuda a comparar meses com quantidades diferentes de dias.

In [ ]:
mensal = df.groupby(["Mes","Mes_nome"]).agg(Venda_total=("Total_Venda","sum"), Unidades=("Quantidade","sum"), Registros=("Total_Venda","size")).reset_index()
mensal["Venda_media_diaria"] = mensal["Venda_total"] / mensal["Data"].nunique() if "Data" in mensal else mensal["Venda_total"] / df.groupby("Mes")["Data"].nunique().values
mensal["Periodo"] = np.where(mensal.Mes < 11, "Antes (jan–out)", np.where(mensal.Mes == 11, "Mês da Black Friday (nov)", "Após (dez)"))
display(mensal[["Mes_nome","Venda_total","Venda_media_diaria","Unidades","Periodo"]])
antes = mensal.loc[mensal.Mes<11,"Venda_media_diaria"].mean(); novembro = mensal.loc[mensal.Mes==11,"Venda_media_diaria"].iloc[0]
print(f"Média diária jan–out: {antes:,.2f} | novembro: {novembro:,.2f} | variação: {(novembro/antes-1):+.1%}")
fig,ax=plt.subplots(figsize=(10,4)); sns.lineplot(data=mensal,x="Mes",y="Venda_media_diaria",marker="o",ax=ax,color="#4E79A7")
ax.axvline(11,color="#E15759",linestyle="--",label="Black Friday em novembro"); ax.set_xticks(range(1,13),mes_ordem)
ax.set_ylabel("Venda média diária"); ax.set_title("Vendas mensais: média diária"); ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f"Conclusão: novembro ficou {abs(novembro/antes-1):.1%} {'acima' if novembro>antes else 'abaixo'} da média diária de janeiro a outubro.")

## 8. Ticket médio por faixa etária e categoria

Sem identificador de compra, usamos a média de `Total_Venda` por linha como **valor médio por registro de produto**. Essa medida não deve ser interpretada como ticket por cliente ou cesta.

In [ ]:
ticket = df.pivot_table(index="Faixa_etaria", columns="Categoria", values="Total_Venda", aggfunc="mean", observed=False)
display(ticket)
print("Média geral por faixa etária (valor por registro):")
display(df.groupby("Faixa_etaria", observed=False)["Total_Venda"].mean().rename("Valor_medio_por_registro").to_frame())
fig,ax=plt.subplots(figsize=(11,5)); sns.heatmap(ticket,annot=True,fmt=".0f",cmap="YlGnBu",ax=ax,cbar_kws={"label":"Valor médio"})
ax.set_title("Valor médio por registro: faixa etária × categoria"); ax.set_xlabel(""); ax.set_ylabel(""); plt.xticks(rotation=35,ha="right"); plt.tight_layout(); plt.show()
print(f"Conclusão: o maior valor médio por registro ocorreu em {ticket.stack().idxmax()} ({ticket.stack().max():,.2f}).")

## 9. Semana da Black Friday versus semana do Natal

Compara-se a semana calendário de segunda a domingo que contém cada data: 20–26/11/2023 e 25–31/12/2023. Indicador principal: venda média diária total da loja.

In [ ]:
bf = pd.Timestamp("2023-11-24"); natal = pd.Timestamp("2023-12-25")
def intervalo_semana(d):
    inicio = d - pd.Timedelta(days=d.weekday()); return inicio, inicio + pd.Timedelta(days=6)
linhas=[]
for nome, data in [("Black Friday",bf),("Natal",natal)]:
    ini,fim=intervalo_semana(data); parte=df[df.Data.between(ini,fim)]; por_data=parte.groupby("Data").Total_Venda.sum()
    linhas.append({"Semana":nome,"Inicio":ini.date(),"Fim":fim.date(),"Venda_total":por_data.sum(),"Media_diaria":por_data.mean(),"Dias_com_dados":por_data.size})
comp=pd.DataFrame(linhas).set_index("Semana")
display(comp)
vencedora=comp.Media_diaria.idxmax(); print(f"Maior média diária: {vencedora} ({comp.loc[vencedora,'Media_diaria']:,.2f}).")
fig,ax=plt.subplots(figsize=(7,4)); sns.barplot(x=comp.index,y=comp.Media_diaria.values,ax=ax)
ax.set(title="Média diária nas semanas da Black Friday e do Natal",ylabel="Valor médio por dia",xlabel=""); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print(f"Conclusão: a semana de {vencedora} superou a outra em {abs(comp.Media_diaria.iloc[0]-comp.Media_diaria.iloc[1]):,.2f} por dia, em média.")

## 10. Impacto observado dos descontos e produtos com maior aumento

Promoção é qualquer registro com desconto maior que zero. O efeito é descritivo: compara registros promocionais e sem promoção. O aumento por produto usa quantidade média diária nos dias em que houve desconto no produto versus os demais dias; exige ao menos um dia em cada grupo.

In [ ]:
impacto = df.groupby("Promocao").agg(Registros=("Total_Venda","size"), Desconto_medio=("Desconto","mean"), Quantidade_media=("Quantidade","mean"), Venda_media=("Total_Venda","mean"), Venda_soma=("Total_Venda","sum"))
impacto.index = impacto.index.map({False:"Sem desconto",True:"Com desconto"})
display(impacto)
por_prod = df.groupby(["Produto","Promocao"])["Quantidade"].mean().unstack()
por_prod = por_prod.dropna(subset=[False,True])
por_prod["Aumento_qtd_media_%"] = np.where(por_prod[False].ne(0),(por_prod[True]/por_prod[False]-1)*100,np.nan)
por_prod["Dias_com_promocao"] = df[df.Promocao].groupby("Produto").size()
display(por_prod.sort_values("Aumento_qtd_media_%",ascending=False).head(15))
print("Interpretação: diferença observacional; não é possível afirmar causalidade.")
fig,axes=plt.subplots(1,2,figsize=(10,4))
sns.barplot(x=impacto.index,y=impacto["Quantidade_media"].values,ax=axes[0]); axes[0].set_title("Quantidade média por registro")
sns.barplot(x=impacto.index,y=impacto["Venda_media"].values,ax=axes[1]); axes[1].set_title("Venda média por registro")
for ax in axes: ax.set_xlabel(""); ax.tick_params(axis="x",rotation=20)
plt.tight_layout(); plt.show()
melhor=por_prod.sort_values("Aumento_qtd_media_%",ascending=False).head(10).sort_values("Aumento_qtd_media_%")
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(x=melhor["Aumento_qtd_media_%"].values,y=melhor.index,ax=ax,color="#59A14F")
ax.set(title="Produtos com maior variação de quantidade em dias promocionais",xlabel="Variação vs. dias sem desconto (%)",ylabel=""); plt.tight_layout(); plt.show()
delta=impacto.loc["Com desconto","Quantidade_media"]-impacto.loc["Sem desconto","Quantidade_media"]
print(f"Conclusão: a quantidade média por registro foi {abs(delta):.2f} unidades {'maior' if delta>0 else 'menor' if delta<0 else 'igual'} em registros promocionais. {por_prod['Aumento_qtd_media_%'].idxmax()} teve a maior alta estimada ({por_prod['Aumento_qtd_media_%'].max():+.1f}%). Associação descritiva, não causal.")

## Conclusão

As tabelas e gráficos acima mostram os resultados por categoria, produto, período e faixa etária. Ao interpretar os resultados, considere que a base contém registros agregados por produto e data, cobre somente um ano e não informa moeda nem transações/cliente identificável. Portanto, padrões temporais e relações com descontos devem ser tratados como evidência descritiva para orientar investigações, não como estimativas causais.